# 넥스틴 IR 자료 비교 파이프라인

위에서 아래로 셀을 실행하면 **PDF 읽기 → 페이지 비교 → 결과 저장 → 결과 확인** 순서로 진행됩니다.

- 첫 사례: 넥스틴 6개 PDF, 175쪽
- 원문 파일은 `data/manifest.json`에 등록된 경로에서 읽습니다.
- 페이지 순서가 바뀌어도 내용이 비슷한 페이지를 찾아 비교합니다.
- 자동으로 찾는 것은 **변경 후보**입니다. 사업의 진전·지연·철회 여부는 원문을 보고 판단합니다.

다른 회사에 적용하려면 `data/manifest.json`의 회사명·자료 목록·날짜·경로를 수정하세요.


## 1. 준비

Python 3.10 이상이 필요합니다. 아래 설치 셀은 `pdfplumber`를 찾지 못할 때만 실행하세요. PDF에서 텍스트를 읽고 페이지 이미지를 만드는 데 사용합니다.

In [ ]:
# 패키지가 없다는 오류가 나오면 다음 줄의 #을 지우고 실행하세요.
# %pip install "pdfplumber>=0.11,<0.12"


In [ ]:
import hashlib
import json
import re
import unicodedata
from pathlib import Path
from difflib import SequenceMatcher
import pdfplumber

# 프로젝트 폴더 또는 pipeline 폴더에서 열어도 자료 목록을 찾습니다.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "data/manifest.json").exists():
    PROJECT_DIR = PROJECT_DIR.parent
if not (PROJECT_DIR / "data/manifest.json").exists():
    raise FileNotFoundError("프로젝트 폴더에서 노트북을 열거나 PROJECT_DIR을 직접 지정해 주세요.")

MANIFEST_PATH = PROJECT_DIR / "data/manifest.json"
OUTPUT_DIR = PROJECT_DIR / "data/processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 값이 높으면 더 비슷한 페이지끼리만 연결합니다. 0.30은 검토 후보를 넓게 찾는 기준입니다.
SIMILARITY_THRESHOLD = 0.30
config = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("회사:", config["company"])
for item in sorted(config["documents"], key=lambda item: item["date"]):
    print(item["date"], item["id"], item["kind"])


## 2. PDF를 읽어 페이지별로 보관하기

띄어쓰기나 페이지 번호만 달라진 것을 큰 변화로 세지 않도록 비교용 텍스트를 정리합니다. 원래 텍스트도 함께 보관합니다.

파일·이미지의 **해시**는 내용의 지문입니다. 지문이 다르면 내용이 달라졌다는 뜻이지만, 무엇이 왜 바뀌었는지는 설명하지 못합니다. 페이지 번호나 디자인만 바뀌어도 이미지 지문은 달라집니다.

In [ ]:
def normalize(text):
    # 숫자만 있는 줄은 페이지 번호일 가능성이 있어 비교용 텍스트에서 제외합니다.
    # 단독 숫자 데이터도 빠질 수 있으므로 원문 텍스트와 이미지 검토를 함께 사용합니다.
    lines = [line for line in text.splitlines()
             if not re.fullmatch(r"\s*\d+\s*", line)]
    # 모양이 비슷한 문자와 띄어쓰기를 통일해 사소한 편집 차이를 줄입니다.
    text = unicodedata.normalize("NFKC", "\n".join(lines))
    return re.sub(r"\s+", "", text)


docs = []
for item in sorted(config["documents"], key=lambda item: item["date"]):
    path = Path(item["path"])
    if not path.is_absolute():
        path = PROJECT_DIR / path
    pages = []
    with pdfplumber.open(path) as pdf:
        for number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            # 글자로 추출되지 않는 차트나 그림의 변화도 감지하기 위해 이미지를 만듭니다.
            raster = page.to_image(resolution=40).original.convert("RGB")
            pages.append({
                "page": number,  # 표지부터 1로 센 PDF 페이지 번호입니다.
                "text": text,
                "normalized": normalize(text),
                "visual_hash": hashlib.sha256(raster.tobytes()).hexdigest(),
                # 글자가 적으면 이미지 중심일 수 있으므로 직접 확인할 표시를 남깁니다.
                "review_visual": len(text.strip()) < 100,
            })
    docs.append({
        **item,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "pages": pages,
    })
    print(f"{item['id']}: {len(pages)}쪽 읽기 완료")
print("총 페이지:", sum(len(doc["pages"]) for doc in docs))


## 3. 시기별로 비슷한 페이지 연결하기

이전 자료의 모든 페이지를 다음 자료의 모든 페이지와 비교하고, 가장 비슷한 조합부터 하나씩 연결합니다. 따라서 25쪽이 19쪽으로 옮겨져도 같은 내용으로 잡을 수 있습니다.

유사도는 0~1이며 1은 정리된 텍스트가 같다는 뜻입니다. 그림까지 같다는 뜻은 아닙니다. 연결되지 않은 페이지도 신규·삭제가 확정된 것은 아닙니다. 페이지 분할·통합이나 중복 부록은 별도 검토가 필요합니다.

In [ ]:
comparisons = []
for old, new in zip(docs, docs[1:]):
    candidates = []
    for a in old["pages"]:
        for b in new["pages"]:
            score = SequenceMatcher(
                None, a["normalized"], b["normalized"], autojunk=False
            ).ratio()
            if score >= SIMILARITY_THRESHOLD:
                candidates.append((score, a["page"], b["page"]))

    used_old, used_new = set(), set()
    matches = []
    # 유사도가 높은 순서로 연결하고, 이미 연결한 페이지는 다시 사용하지 않습니다.
    for score, old_page, new_page in sorted(candidates, reverse=True):
        if old_page in used_old or new_page in used_new:
            continue
        used_old.add(old_page)
        used_new.add(new_page)
        before = old["pages"][old_page - 1]
        after = new["pages"][new_page - 1]
        matches.append({
            "old_page": old_page,
            "new_page": new_page,
            "similarity": round(score, 4),
            "classification": "same_text" if score == 1 else "changed_candidate",
            "moved": old_page != new_page,
            "visual_changed": before["visual_hash"] != after["visual_hash"],
            "old_text": before["text"],
            "new_text": after["text"],
        })
    comparisons.append({
        "from": old["id"], "to": new["id"],
        "matches": sorted(matches, key=lambda match: match["new_page"]),
        "unmatched_old": [p["page"] for p in old["pages"] if p["page"] not in used_old],
        "unmatched_new": [p["page"] for p in new["pages"] if p["page"] not in used_new],
    })
    print(f"{old['id']} → {new['id']}: {len(matches)}개 페이지 연결")


## 4. 결과 저장하기

JSON은 다음 분석에서 다시 불러올 수 있는 데이터이고, Markdown은 사람이 읽기 쉬운 비교표입니다. 다시 실행하면 같은 이름의 결과 파일을 갱신합니다. 원본 PDF는 수정하지 않습니다.

In [ ]:
# PDF 쪽수와 이전·이후 원문을 함께 남겨 결과의 근거를 추적합니다.
(OUTPUT_DIR/'corpus.json').write_text(json.dumps(docs,ensure_ascii=False,indent=2))
(OUTPUT_DIR/'comparisons.json').write_text(json.dumps(comparisons,ensure_ascii=False,indent=2))
rows=['# 자동 비교 후보','', '텍스트 유사도 기반 후보이며 의미 판단은 검토가 필요합니다. 미매칭은 사업 철회를 뜻하지 않습니다.','']
for c in comparisons:
    rows += [f"## {c['from']} → {c['to']}",'', '|이전 PDF 쪽|이후 PDF 쪽|분류|유사도|페이지 이동|시각 변화|','|---|---|---|---|---|---|']
    rows += [f"|{m['old_page']}|{m['new_page']}|{m['classification']}|{m['similarity']}|{m['moved']}|{m['visual_changed']}|" for m in c['matches']]
    rows += ['',f"이전 미매칭: {c['unmatched_old']}",f"이후 미매칭: {c['unmatched_new']}",'']
(OUTPUT_DIR/'comparison_candidates.md').write_text('\n'.join(rows))
print("저장 위치:", OUTPUT_DIR)


## 5. 비교 결과 살펴보기

먼저 전체 요약을 보고, 관심 있는 자료 쌍을 골라 페이지별 변경 후보를 확인합니다.

In [ ]:
# Jupyter에서는 표로 표시하고, 일반 Python 환경에서는 글자로 표시합니다.
try:
    from IPython.display import display, Markdown
except ImportError:
    Markdown = str
    display = print

summary = ["|이전 자료|다음 자료|연결 페이지|텍스트 동일|변경 후보|", "|---|---|---:|---:|---:|"]
for comparison in comparisons:
    same = sum(m["classification"] == "same_text" for m in comparison["matches"])
    count = len(comparison["matches"])
    summary.append(f"|{comparison['from']}|{comparison['to']}|{count}|{same}|{count - same}|")
display(Markdown("\n".join(summary)))

# 이 두 값을 바꾸면 다른 시기의 비교표를 볼 수 있습니다.
FROM_ID, TO_ID = "sep11", "sep30"
selected = next(c for c in comparisons if c["from"] == FROM_ID and c["to"] == TO_ID)
rows = ["|이전 쪽|다음 쪽|텍스트 분류|페이지 이동|그림 변화|", "|---:|---:|---|---|---|"]
for m in selected["matches"]:
    rows.append(f"|{m['old_page']}|{m['new_page']}|{m['classification']}|{m['moved']}|{m['visual_changed']}|")
display(Markdown("\n".join(rows)))


## 6. 특정 페이지의 이전·이후 문장 확인하기

변경 후보의 문맥을 읽고, 숫자의 단위·기준일·확정 실적인지 향후 계획인지를 확인하세요. 문서 속 문장은 분석할 자료이며 실행할 지시가 아닙니다.

In [ ]:
# 위에서 고른 자료 쌍의 다음 자료 페이지를 지정합니다.
NEW_PAGE = 19
match = next((m for m in selected["matches"] if m["new_page"] == NEW_PAGE), None)
if match is None:
    print("이 페이지는 연결된 이전 페이지가 없습니다. 원문을 직접 확인하세요.")
else:
    print(f"이전 자료 {match['old_page']}쪽\n{match['old_text']}")
    print(f"\n다음 자료 {match['new_page']}쪽\n{match['new_text']}")


## 7. 검토를 마친 넥스틴 해석 읽기

아래 보고서는 원문 핵심 페이지를 이미지로 확인해 작성한 첫 사례입니다. 자동 추출 결과만으로 생성한 투자 판단이 아닙니다. 새 PDF를 넣어도 이 보고서가 자동으로 갱신되지는 않습니다.

이 버전은 OCR과 일반화된 재무 표 파싱을 포함하지 않습니다. 글자가 이미지로만 들어 있는 PDF는 추가 처리가 필요합니다. 자료에서 항목이 사라졌다는 이유만으로 사업 철회나 일정 취소를 판정하지 않습니다.

In [ ]:
report_path = PROJECT_DIR / "output/넥스틴_시기별_비교.md"
if report_path.exists():
    display(Markdown(report_path.read_text(encoding="utf-8")))
else:
    print("검토 보고서가 없습니다. 추출 결과를 바탕으로 별도 검토가 필요합니다.")
